In [6]:
import openai, json, requests, os
from dotenv import load_dotenv

load_dotenv
client = openai.OpenAI()
messages = []

In [7]:
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return json.dumps(response.json())

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return json.dumps(response.json())

def get_similar_movies(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return json.dumps(response.json())

FUNCTION_MAP = {
      "get_popular_movies": get_popular_movies,
      "get_movie_details": get_movie_details,
      "get_similar_movies": get_similar_movies,
  }


In [8]:
TOOLS = [
      {
          "type": "function",
          "function": {
              "name": "get_popular_movies",
              "description": "Get a list of currently popular movies.",
              "parameters": {
                  "type": "object",
                  "properties": {},
                  "required": [],
              },
          },
      },
      {
          "type": "function",
          "function": {
              "name": "get_movie_details",
              "description": "Get detailed information about a specific movie by its ID.",
              "parameters": {
                  "type": "object",
                  "properties": {
                      "id": {
                          "type": "string",
                          "description": "The movie ID.",
                      }
                  },
                  "required": ["id"],
              },
          },
      },
      {
          "type": "function",
          "function": {
              "name": "get_similar_movies",
              "description": "Get movies similar to a specific movie by its ID.",
              "parameters": {
                  "type": "object",
                  "properties": {
                      "id": {
                          "type": "string",
                          "description": "The movie ID.",
                      }
                  },
                  "required": ["id"],
              },
          },
      },
  ]

In [9]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [10]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 요즘 핫한 영화 좀 알려줘ㅜ
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {} for a result of [{"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/1x9e0qWonw634NhIsRdvnneeqvN.jpg", "genre_ids": [10749, 18], "id": 1523145, "original_language": "ru", "original_title": "\u0422\u0432\u043e\u0451 \u0441\u0435\u0440\u0434\u0446\u0435 \u0431\u0443\u0434\u0435\u0442 \u0440\u0430\u0437\u0431\u0438\u0442\u043e", "overview": "High school student Polina is saved from bullying at her new school and makes a deal with the main bully Bars: he must pretend to be her boyfriend and protect her, and she must do everything he says. During this game, the couple develops real feelings, but her family and classmates have reasons to separate the lovers.", "popularity": 929.5163, "poster_path": "https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg", "release_date": "2026-03-26", "title": "Your Heart Will Be Broken", "video": false, "vote_average": 6.75, "vote_